# Devreotes Lab GraphRAG — Knowledge Graph Loading

**Sylvester Vhashe & Tadiwanashe C. Dzimbanete**

DAV 6500 Capstone | Katz School, Yeshiva University | 2026

This notebook loads the knowledge graph into Neo4j Aura from our Cypher backup file. We used this after building the entity extraction pipeline to populate the graph, and again whenever we needed to rebuild after an Aura Free instance expired.

In [ ]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 6.9 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving load_graph.cypher to load_graph.cypher


In [ ]:
import os
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ.get("NEO4J_URI", "neo4j+s://YOUR_INSTANCE.databases.neo4j.io"),
    auth=(os.environ.get("NEO4J_USER", "YOUR_USER"), os.environ.get("NEO4J_PASSWORD", "YOUR_PASSWORD"))
)

# Load graph
print("Loading graph...")
with open("load_graph.cypher", "r") as f:
    statements = [line.strip() for line in f if line.strip()]

print(f"Found {len(statements)} statements")
errors = []
with driver.session() as session:
    for i, stmt in enumerate(statements):
        try:
            session.run(stmt)
        except Exception as e:
            errors.append(f"Line {i+1}: {str(e)[:80]}")
        if (i+1) % 500 == 0:
            print(f"  Loaded {i+1}/{len(statements)}...")

print(f"Done: {len(statements)} loaded, {len(errors)} errors")

# Remove duplicates
print("Removing duplicates...")
with driver.session() as session:
    for num in ['271', '090', '034']:
        session.run("MATCH (p:Paper {number: $n}) DETACH DELETE p", n=num)
        print(f"  Deleted #{num}")

# Add Gene nodes
print("Creating Gene nodes...")
with driver.session() as session:
    records = list(session.run("MATCH (p:Paper)-[r:MENTIONS_PROTEIN]->(pr:Protein) RETURN p.number AS pn, pr.name AS name, r.context AS ctx"))
    for i, r in enumerate(records):
        session.run("MERGE (g:Gene {name:$name}) WITH g MATCH (p:Paper {number:$pn}) MERGE (p)-[:MENTIONS_GENE {context:$ctx}]->(g)", name=r['name'], pn=r['pn'], ctx=r['ctx'] or '')
        if (i+1) % 300 == 0:
            print(f"  {i+1}/{len(records)}...")
    print(f"  Created {len(records)} gene connections")

# Verify
print("\nFinal graph:")
with driver.session() as session:
    for label in ['Paper','Protein','Gene','Method','Concept','Organism']:
        c = session.run(f"MATCH (n:{label}) RETURN count(n) AS c").single()["c"]
        print(f"  {label}: {c}")

driver.close()
print("Done!")

Loading graph...
Found 3887 statements
  Loaded 500/3887...
  Loaded 1000/3887...
  Loaded 1500/3887...
  Loaded 2000/3887...
  Loaded 2500/3887...
  Loaded 3000/3887...
  Loaded 3500/3887...
Done: 3887 loaded, 0 errors
Removing duplicates...
  Deleted #271
  Deleted #090
  Deleted #034
Creating Gene nodes...
  300/1497...
  600/1497...
  900/1497...
  1200/1497...
  Created 1497 gene connections

Final graph:
  Paper: 235
  Protein: 738
  Gene: 738
  Method: 531
  Concept: 330
  Organism: 228
Done!
